# Urban Sound Classification — ANN vs SVM (v3)

This notebook classifies UrbanSound8K audio clips using an Artificial Neural
Network (tuned with KerasTuner) and a Support Vector Machine baseline.

**Key methodological fixes over the previous version:**
- No data leakage: a held-out **test set is used exactly once**, at the end.
- Proper **70% / 15% / 15%** train / validation / test split, **stratified** by class.
- Global **random seeds** for `random`, `numpy`, and `tensorflow`.
- **StandardScaler** fit on the training set only, reused for val/test/SVM/inference.
- All artifacts saved: model (`.keras`), scaler, label encoder, history, metrics.
- Full **precision / recall / F1 / confusion matrix** reporting, not just accuracy.
- Data augmentation (noise, gain, time shift, pitch shift, time stretch) **applied
  only to the training split**.
- Code organized into reusable functions with error handling for missing/corrupted files.

## Notebook structure
1. Imports
2. Random Seeds
3. Load Dataset
4. EDA
5. Feature Extraction
6. Split Dataset (+ Label Encoding + Data Augmentation)
7. Feature Scaling
8. ANN Model
9. Hyperparameter Tuning
10. Train Best ANN
11. Evaluate ANN
12. Train SVM
13. Compare Models
14. Save Models
15. Predict New Audio


## 1. Imports

In [ ]:
!pip install -q keras-tuner soundfile


In [ ]:
import json
import os
import pickle
import random
import time
import warnings
from pathlib import Path

import joblib
import keras_tuner
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import regularizers
from tensorflow.keras.utils import to_categorical
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")


## 2. Random Seeds\n\nSet every source of randomness so results are reproducible end to end.

In [ ]:
SEED = 42


def set_global_seed(seed: int = SEED) -> None:
    """Seed python, numpy and tensorflow RNGs for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_global_seed(SEED)


## 3. Load Dataset

In [ ]:
DATASET_DIR = Path("/kaggle/input/datasets/mayishataltabmridu/urban-sound/Urban Sound")
METADATA_PATH = DATASET_DIR / "/kaggle/input/datasets/mayishataltabmridu/urban-sound/Urban Sound/UrbanSound8K.csv"
ARTIFACTS_DIR = Path("/kaggle/working/saved_models")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


def load_metadata(metadata_path: Path = METADATA_PATH) -> pd.DataFrame:
    if not metadata_path.exists():
        raise FileNotFoundError(f"Metadata file not found at: {metadata_path}")
    return pd.read_csv(metadata_path)


metadata = load_metadata()
print(f"Dataset shape: {metadata.shape}")
metadata.head()

In [ ]:
def get_file_path(row: pd.Series, dataset_dir: Path = DATASET_DIR) -> Path:
    return dataset_dir / f"fold{row['fold']}" / row["slice_file_name"]


### 3.1 Automatic Dataset Check

Before doing any expensive feature extraction, verify the dataset is sound:
missing files, duplicate entries, and corrupted/unreadable audio.

In [ ]:
def check_dataset(df: pd.DataFrame, dataset_dir: Path = DATASET_DIR) -> dict:
    """Validate dataset integrity before training.

    Checks for missing audio files, duplicate metadata rows, and
    corrupted/unreadable audio files (via a fast header-only read).
    """
    issues = {"missing_files": [], "duplicates": [], "corrupted_files": []}

    dup_mask = df.duplicated(subset=["slice_file_name", "fold"], keep=False)
    issues["duplicates"] = df.loc[dup_mask, "slice_file_name"].tolist()

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Checking dataset integrity"):
        file_path = get_file_path(row, dataset_dir)
        if not file_path.exists():
            issues["missing_files"].append(str(file_path))
            continue
        try:
            sf.info(str(file_path))
        except Exception:
            issues["corrupted_files"].append(str(file_path))

    print(f"Missing files:   {len(issues['missing_files'])}")
    print(f"Duplicate rows:  {len(issues['duplicates'])}")
    print(f"Corrupted files: {len(issues['corrupted_files'])}")
    return issues


dataset_issues = check_dataset(metadata)


## 4. EDA

In [ ]:
def plot_class_distribution(df: pd.DataFrame) -> None:
    plt.figure(figsize=(16, 5))
    counts = df["class"].value_counts()

    plt.subplot(1, 2, 1)
    sns.barplot(x=counts.index, y=counts.values, palette="Set2")
    plt.title("Class Distribution (Count)", fontweight="black", size=13)
    for i, v in enumerate(counts.values):
        plt.text(i, v, v, ha="center", fontsize=10)
    plt.xticks(rotation=90)

    plt.subplot(1, 2, 2)
    colors = sns.color_palette("Set2", len(counts))
    plt.pie(counts, labels=counts.index, autopct="%.1f%%", textprops={"size": 11}, colors=colors, startangle=90)
    plt.title("Class Distribution (%)", fontweight="black", size=13)

    plt.tight_layout()
    plt.show()


plot_class_distribution(metadata)


In [ ]:
def plot_audio_overview(file_path: Path, sr: int = 22050, n_mfcc: int = 40) -> None:
    """Plot waveform, spectrogram, mel-spectrogram and MFCCs for one audio file."""
    audio, sample_rate = librosa.load(file_path, sr=sr)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    librosa.display.waveshow(audio, sr=sample_rate, ax=axes[0, 0])
    axes[0, 0].set_title("Waveform")

    stft_db = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
    img1 = librosa.display.specshow(stft_db, sr=sample_rate, x_axis="time", y_axis="log", ax=axes[0, 1])
    axes[0, 1].set_title("Spectrogram")
    fig.colorbar(img1, ax=axes[0, 1], format="%+2.0f dB")

    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sample_rate)
    mel_db = librosa.power_to_db(mel_spec, ref=np.max)
    img2 = librosa.display.specshow(mel_db, sr=sample_rate, x_axis="time", y_axis="mel", ax=axes[1, 0])
    axes[1, 0].set_title("Mel Spectrogram")
    fig.colorbar(img2, ax=axes[1, 0], format="%+2.0f dB")

    mfcc = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc)
    img3 = librosa.display.specshow(mfcc, sr=sample_rate, x_axis="time", ax=axes[1, 1])
    axes[1, 1].set_title("MFCC")
    fig.colorbar(img3, ax=axes[1, 1])

    plt.suptitle(file_path.name)
    plt.tight_layout()
    plt.show()


sample_row = metadata.iloc[0]
plot_audio_overview(get_file_path(sample_row))


## 5. Feature Extraction

Features are **MFCC mean + MFCC std** (concatenated), which captures both the
average spectral shape and how much it varies over time. Data augmentation
functions are defined here but are only ever applied to the *training* split
(see section 6.2), never to validation/test data.

In [ ]:
def add_noise(audio: np.ndarray, noise_factor: float = 0.005) -> np.ndarray:
    noise = np.random.randn(len(audio))
    return audio + noise_factor * noise


def apply_gain(audio: np.ndarray, gain_range: tuple = (0.8, 1.2)) -> np.ndarray:
    gain = np.random.uniform(*gain_range)
    return audio * gain


def time_shift(audio: np.ndarray, shift_max: float = 0.2) -> np.ndarray:
    shift = int(np.random.uniform(-shift_max, shift_max) * len(audio))
    return np.roll(audio, shift)


def pitch_shift(audio: np.ndarray, sr: int, n_steps_range: tuple = (-2, 2)) -> np.ndarray:
    n_steps = np.random.uniform(*n_steps_range)
    return librosa.effects.pitch_shift(y=audio, sr=sr, n_steps=n_steps)


def time_stretch(audio: np.ndarray, rate_range: tuple = (0.8, 1.2)) -> np.ndarray:
    rate = np.random.uniform(*rate_range)
    return librosa.effects.time_stretch(y=audio, rate=rate)


def augment_audio(audio: np.ndarray, sr: int) -> np.ndarray:
    """Randomly apply a subset of augmentations. Training data only."""
    augmented = audio.copy()
    if np.random.rand() < 0.5:
        augmented = add_noise(augmented)
    if np.random.rand() < 0.3:
        augmented = apply_gain(augmented)
    if np.random.rand() < 0.3:
        augmented = time_shift(augmented)
    if np.random.rand() < 0.2:
        try:
            augmented = pitch_shift(augmented, sr)
        except Exception:
            pass
    if np.random.rand() < 0.2:
        try:
            augmented = time_stretch(augmented)
        except Exception:
            pass
    return augmented


In [ ]:
def extract_features(file_path: Path, n_mfcc: int = 40, augment: bool = False, sr: int = 22050):
    """Load an audio file and return concatenated MFCC mean+std features.

    Returns ``None`` (instead of raising) when the file is missing, empty,
    or otherwise unreadable/corrupted, so that a caller can skip it.
    """
    file_path = Path(file_path)
    if not file_path.exists():
        warnings.warn(f"Missing file: {file_path}")
        return None

    try:
        audio, sample_rate = librosa.load(file_path, sr=sr)
    except Exception as exc:
        warnings.warn(f"Could not read {file_path}: {exc}")
        return None

    if audio is None or len(audio) == 0:
        warnings.warn(f"Empty/corrupted audio file: {file_path}")
        return None

    if augment:
        audio = augment_audio(audio, sample_rate)

    try:
        mfcc = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc)
    except Exception as exc:
        warnings.warn(f"Feature extraction failed for {file_path}: {exc}")
        return None

    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)
    return np.concatenate([mfcc_mean, mfcc_std])


In [ ]:
N_MFCC = 40


def extract_all_features(df: pd.DataFrame, dataset_dir: Path = DATASET_DIR, n_mfcc: int = N_MFCC) -> pd.DataFrame:
    """Extract base (non-augmented) features for every row, keeping file paths
    around so later steps can re-derive augmented versions for the train split."""
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting MFCC features"):
        file_path = get_file_path(row, dataset_dir)
        feat = extract_features(file_path, n_mfcc=n_mfcc, augment=False)
        if feat is None:
            continue
        records.append({"file_path": file_path, "class": row["class"], "features": feat})
    return pd.DataFrame(records)


features_df = extract_all_features(metadata, n_mfcc=N_MFCC)
print(f"Successfully extracted features for {len(features_df)} / {len(metadata)} files")


## 6. Split Dataset

A **70% / 15% / 15%** train / validation / test split, **stratified by class**
so the class balance is preserved in every split. The test set is set aside
here and is not touched again until section 11 (final evaluation).

In [ ]:
train_df, temp_df = train_test_split(
    features_df,
    test_size=0.30,
    random_state=SEED,
    stratify=features_df["class"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["class"],
)

print(f"Train: {len(train_df)} ({len(train_df) / len(features_df):.1%})")
print(f"Val:   {len(val_df)} ({len(val_df) / len(features_df):.1%})")
print(f"Test:  {len(test_df)} ({len(test_df) / len(features_df):.1%})")

X_val = np.stack(val_df["features"].to_numpy())
X_test = np.stack(test_df["features"].to_numpy())
y_val_raw = val_df["class"].to_numpy()
y_test_raw = test_df["class"].to_numpy()

X_train_base = np.stack(train_df["features"].to_numpy())
y_train_base_raw = train_df["class"].to_numpy()


### 6.1 Data Augmentation (Training Split Only)

Additional augmented copies (noise injection, gain, time shift, pitch shift,
time stretch) are generated **only from the training audio files**, then
concatenated with the original training features. Validation and test data
are never augmented.

In [ ]:
def augment_training_set(df: pd.DataFrame, n_augmentations: int = 1, n_mfcc: int = N_MFCC):
    aug_features, aug_labels = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Augmenting training data"):
        for _ in range(n_augmentations):
            feat = extract_features(row["file_path"], n_mfcc=n_mfcc, augment=True)
            if feat is not None:
                aug_features.append(feat)
                aug_labels.append(row["class"])
    return np.array(aug_features), np.array(aug_labels)


X_train_aug, y_train_aug_raw = augment_training_set(train_df, n_augmentations=1)

X_train = np.concatenate([X_train_base, X_train_aug], axis=0)
y_train_raw = np.concatenate([y_train_base_raw, y_train_aug_raw], axis=0)
print(f"Training set size after augmentation: {X_train.shape[0]} (was {X_train_base.shape[0]})")


### 6.2 Label Encoding\n\nThe encoder is fit on the full set of classes so val/test labels are always covered.

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(metadata["class"])
class_names = list(label_encoder.classes_)
NUM_CLASSES = len(class_names)

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Label mapping:", label_mapping)

y_train_int = label_encoder.transform(y_train_raw)
y_val_int = label_encoder.transform(y_val_raw)
y_test_int = label_encoder.transform(y_test_raw)

y_train = to_categorical(y_train_int, num_classes=NUM_CLASSES)
y_val = to_categorical(y_val_int, num_classes=NUM_CLASSES)
y_test = to_categorical(y_test_int, num_classes=NUM_CLASSES)


## 7. Feature Scaling

`StandardScaler` is fit **only on the training features** and reused
(via `.transform`) on validation, test, SVM, and future single-file
predictions. This scaler is saved in section 14 so inference never
re-fits it on new data.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

INPUT_DIM = X_train_scaled.shape[1]
print(f"Feature dimensionality: {INPUT_DIM}")


## 8. ANN Model

Architecture: `Dense -> BatchNorm -> ReLU -> Dropout`, with L2 regularization
on the dense layers. Defined as a `keras_tuner.HyperModel` so both the
architecture *and* the training batch size are searchable hyperparameters.

In [ ]:
class ANNHyperModel(keras_tuner.HyperModel):
    def build(self, hp):
        model = keras.Sequential()
        model.add(keras.Input(shape=(INPUT_DIM,)))

        num_layers = hp.Int("num_layers", min_value=1, max_value=4, step=1)
        l2_reg = hp.Choice("l2_reg", values=[0.0, 1e-4, 1e-3, 1e-2])
        dropout_rate = hp.Float("dropout_rate", min_value=0.1, max_value=0.5, step=0.1)

        for i in range(num_layers):
            units = hp.Int(f"units_{i}", min_value=32, max_value=512, step=32)
            model.add(keras.layers.Dense(units, kernel_regularizer=regularizers.l2(l2_reg)))
            model.add(keras.layers.BatchNormalization())
            model.add(keras.layers.Activation("relu"))
            model.add(keras.layers.Dropout(dropout_rate))

        model.add(keras.layers.Dense(NUM_CLASSES, activation="softmax"))

        learning_rate = hp.Choice("learning_rate", values=[1e-2, 1e-3, 5e-4, 1e-4])
        optimizer_name = hp.Choice("optimizer", values=["adam", "rmsprop"])
        optimizer = (
            keras.optimizers.Adam(learning_rate=learning_rate)
            if optimizer_name == "adam"
            else keras.optimizers.RMSprop(learning_rate=learning_rate)
        )

        model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"])
        return model

    def fit(self, hp, model, *args, **kwargs):
        batch_size = hp.Choice("batch_size", values=[16, 32, 64, 128])
        return model.fit(*args, batch_size=batch_size, **kwargs)


## 9. Hyperparameter Tuning

The tuner is validated against the **validation set only** &mdash; the test
set is never seen during tuning, fixing the data-leakage issue from the
previous version.

In [ ]:
tuner = keras_tuner.RandomSearch(
    hypermodel=ANNHyperModel(),
    objective="val_accuracy",
    max_trials=15,
    max_retries_per_trial=2,
    seed=SEED,
    directory="keras_tuner_runs",
    project_name="urban_sound_ann_v3",
    overwrite=True,
)

tuner_callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
]

tuner.search(
    X_train_scaled,
    y_train,
    epochs=50,
    validation_data=(X_val_scaled, y_val),
    callbacks=tuner_callbacks,
    verbose=1,
)

best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best hyperparameters:", best_hp.values)


## 10. Train Best ANN

The final model is retrained with the best hyperparameters using the full
callback stack (`EarlyStopping` with `restore_best_weights=True`,
`ReduceLROnPlateau`, `ModelCheckpoint`), still validating against the
validation set only.

In [ ]:
BEST_MODEL_PATH = ARTIFACTS_DIR / "best_model.keras"


def train_model(model, X_train, y_train, X_val, y_val, model_path: Path, epochs: int = 150, batch_size: int = 32):
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
        keras.callbacks.ModelCheckpoint(filepath=str(model_path), monitor="val_accuracy", save_best_only=True),
    ]
    start = time.time()
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1,
    )
    training_time = time.time() - start
    return history, training_time


best_batch_size = best_hp.values.get("batch_size", 32)
model = tuner.hypermodel.build(best_hp)
model.summary()

history, ann_training_time = train_model(
    model,
    X_train_scaled,
    y_train,
    X_val_scaled,
    y_val,
    model_path=BEST_MODEL_PATH,
    epochs=150,
    batch_size=best_batch_size,
)
print(f"ANN training time: {ann_training_time:.2f}s")

# Reload the checkpointed best-on-val-accuracy model to be safe.
model = keras.models.load_model(BEST_MODEL_PATH)


In [ ]:
def plot_training_history(history) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history["accuracy"], color="red", label="Train Accuracy")
    axes[0].plot(history.history["val_accuracy"], color="blue", label="Val Accuracy")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(history.history["loss"], color="red", label="Train Loss")
    axes[1].plot(history.history["val_loss"], color="blue", label="Val Loss")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()


plot_training_history(history)


## 11. Evaluate ANN

This is the **first and only** time the test set is used. Reports accuracy,
precision/recall/F1 (macro & weighted), a full classification report, a
confusion matrix, per-class accuracy, training time, inference time and
parameter count.

In [ ]:
def evaluate_model(model, X_test, y_test, class_names, model_name: str = "Model", is_keras: bool = True) -> dict:
    """Evaluate a fitted model on held-out test data and report a full metric suite."""
    start = time.time()
    if is_keras:
        y_pred_probs = model.predict(X_test, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
    else:
        y_pred = model.predict(X_test)
    inference_time = time.time() - start

    y_true = np.argmax(y_test, axis=1) if np.ndim(y_test) > 1 else np.asarray(y_test)

    acc = accuracy_score(y_true, y_pred)
    precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    per_class_acc = cm.diagonal() / cm.sum(axis=1)

    print(f"\n=== {model_name} Evaluation (test set) ===")
    print(f"Accuracy:          {acc:.4f}")
    print(f"Precision (macro): {precision_macro:.4f}")
    print(f"Recall (macro):    {recall_macro:.4f}")
    print(f"F1 (macro):        {f1_macro:.4f}")
    print(f"F1 (weighted):     {f1_weighted:.4f}")
    print(f"Inference time:    {inference_time:.4f}s for {len(X_test)} samples")
    print("\nClassification Report:\n", report)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"{model_name} Confusion Matrix")
    plt.tight_layout()
    plt.show()

    return {
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "inference_time": inference_time,
        "confusion_matrix": cm.tolist(),
        "per_class_accuracy": dict(zip(class_names, per_class_acc.tolist())),
    }


def plot_per_class_accuracy(per_class_acc: dict, title: str = "Per-Class Accuracy") -> None:
    plt.figure(figsize=(12, 5))
    sns.barplot(x=list(per_class_acc.keys()), y=list(per_class_acc.values()), palette="viridis")
    plt.xticks(rotation=90)
    plt.ylim(0, 1)
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
ann_metrics = evaluate_model(model, X_test_scaled, y_test, class_names, model_name="ANN")
ann_metrics["training_time"] = ann_training_time
ann_metrics["param_count"] = model.count_params()

plot_per_class_accuracy(ann_metrics["per_class_accuracy"], "ANN Per-Class Accuracy")


## 12. Train SVM

The SVM baseline uses the **same scaled features** as the ANN (never raw
MFCCs). Hyperparameters are tuned with `GridSearchCV`'s internal
cross-validation on train+validation; the test set is still held out and
only used for the final evaluation below.

In [ ]:
X_train_svm = np.concatenate([X_train_scaled, X_val_scaled], axis=0)
y_train_svm = np.concatenate([y_train_int, y_val_int], axis=0)

param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", 0.01, 0.001],
}

svm_grid = GridSearchCV(
    SVC(kernel="rbf", random_state=SEED),
    param_grid,
    cv=5,
    n_jobs=-1,
    scoring="accuracy",
    verbose=0,
)

start = time.time()
svm_grid.fit(X_train_svm, y_train_svm)
svm_training_time = time.time() - start

svm_best = svm_grid.best_estimator_
print("Best SVM params:", svm_grid.best_params_)
print(f"SVM training time: {svm_training_time:.2f}s")


In [ ]:
class SVMWrapper:
    """Thin wrapper so evaluate_model() can treat sklearn and Keras models alike."""

    def __init__(self, estimator):
        self.estimator = estimator

    def predict(self, X):
        return self.estimator.predict(X)


svm_metrics = evaluate_model(
    SVMWrapper(svm_best), X_test_scaled, y_test_int, class_names, model_name="SVM (RBF)", is_keras=False
)
svm_metrics["training_time"] = svm_training_time
svm_metrics["param_count"] = int(svm_best.n_support_.sum())

plot_per_class_accuracy(svm_metrics["per_class_accuracy"], "SVM Per-Class Accuracy")


## 13. Compare Models

In [ ]:
comparison_df = pd.DataFrame(
    [
        {
            "Model": "ANN",
            "Accuracy": ann_metrics["accuracy"],
            "Precision (macro)": ann_metrics["precision_macro"],
            "Recall (macro)": ann_metrics["recall_macro"],
            "F1 (macro)": ann_metrics["f1_macro"],
            "F1 (weighted)": ann_metrics["f1_weighted"],
            "Training Time (s)": ann_metrics["training_time"],
            "Inference Time (s)": ann_metrics["inference_time"],
            "Parameters": ann_metrics["param_count"],
        },
        {
            "Model": "SVM (RBF)",
            "Accuracy": svm_metrics["accuracy"],
            "Precision (macro)": svm_metrics["precision_macro"],
            "Recall (macro)": svm_metrics["recall_macro"],
            "F1 (macro)": svm_metrics["f1_macro"],
            "F1 (weighted)": svm_metrics["f1_weighted"],
            "Training Time (s)": svm_metrics["training_time"],
            "Inference Time (s)": svm_metrics["inference_time"],
            "Parameters": svm_metrics["param_count"],
        },
    ]
)

display(comparison_df)
comparison_df.to_csv(ARTIFACTS_DIR / "ann_svm_comparison.csv", index=False)

plt.figure(figsize=(8, 5))
bars = plt.bar(comparison_df["Model"], comparison_df["Accuracy"], color=["#4C72B0", "#DD8452"])
plt.ylabel("Test Accuracy")
plt.ylim(0, 1)
plt.title("ANN vs SVM \u2014 Test Accuracy")
for bar, v in zip(bars, comparison_df["Accuracy"]):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 0.01, f"{v:.3f}", ha="center")
plt.show()


## 14. Save Models

Every artifact needed to reproduce predictions later is saved: the ANN
model (already checkpointed as `best_model.keras`), the scaler, the label
encoder, the SVM model, the training history, and a metrics summary.

In [ ]:
joblib.dump(scaler, ARTIFACTS_DIR / "scaler.pkl")
joblib.dump(label_encoder, ARTIFACTS_DIR / "label_encoder.pkl")
joblib.dump(svm_best, ARTIFACTS_DIR / "svm_model.pkl")

with open(ARTIFACTS_DIR / "history.pkl", "wb") as f:
    pickle.dump(history.history, f)

metrics_summary = {
    "ann": {k: v for k, v in ann_metrics.items() if k != "confusion_matrix"},
    "svm": {k: v for k, v in svm_metrics.items() if k != "confusion_matrix"},
    "ann_confusion_matrix": ann_metrics["confusion_matrix"],
    "svm_confusion_matrix": svm_metrics["confusion_matrix"],
    "best_hyperparameters": best_hp.values,
    "feature_config": {"n_mfcc": N_MFCC, "feature_dim": INPUT_DIM, "features": "mfcc_mean + mfcc_std"},
    "split_sizes": {"train": int(len(X_train)), "val": int(len(X_val)), "test": int(len(X_test))},
}

with open(ARTIFACTS_DIR / "metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2, default=str)

print("Saved artifacts to:", ARTIFACTS_DIR)
for p in sorted(ARTIFACTS_DIR.glob("*")):
    print(" -", p.name)


## 15. Predict New Audio

The prediction pipeline mirrors training exactly:
**Audio -> MFCC (mean+std) -> Scaler -> ANN -> Label Decoder**.

In [ ]:
def load_prediction_pipeline(artifacts_dir: Path = ARTIFACTS_DIR):
    """Reload every artifact needed for inference from disk."""
    pipeline_model = keras.models.load_model(artifacts_dir / "best_model.keras")
    pipeline_scaler = joblib.load(artifacts_dir / "scaler.pkl")
    pipeline_label_encoder = joblib.load(artifacts_dir / "label_encoder.pkl")
    return pipeline_model, pipeline_scaler, pipeline_label_encoder


def predict_audio(file_path, model, scaler, label_encoder, n_mfcc: int = N_MFCC):
    """Run the full prediction pipeline on a single audio file.

    Raises FileNotFoundError/ValueError for missing or corrupted input so
    callers can handle bad files explicitly instead of getting a silent
    wrong prediction.
    """
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Audio file not found: {file_path}")

    features = extract_features(file_path, n_mfcc=n_mfcc, augment=False)
    if features is None:
        raise ValueError(f"Could not extract features from {file_path} (corrupted or unreadable file).")

    features = features.reshape(1, -1)
    features_scaled = scaler.transform(features)

    probs = model.predict(features_scaled, verbose=0)
    predicted_idx = np.argmax(probs, axis=1)
    predicted_label = label_encoder.inverse_transform(predicted_idx)[0]
    confidence = float(np.max(probs))
    return predicted_label, confidence


In [ ]:
# Example: predict on a held-out sample using the in-memory model/scaler/encoder.
sample_file = DATASET_DIR / "fold7" / "157940-9-0-4.wav"
predicted_label, confidence = predict_audio(sample_file, model=model, scaler=scaler, label_encoder=label_encoder)
print(f"Predicted class: {predicted_label} (confidence: {confidence:.4f})")

# Example: predict using artifacts reloaded fresh from disk (simulates a new session).
loaded_model, loaded_scaler, loaded_label_encoder = load_prediction_pipeline()
predicted_label, confidence = predict_audio(
    sample_file, model=loaded_model, scaler=loaded_scaler, label_encoder=loaded_label_encoder
)
print(f"[from disk] Predicted class: {predicted_label} (confidence: {confidence:.4f})")
